In [ ]:
# Imports.

from pathlib import Path
from time import perf_counter, process_time

import h5py
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


In [ ]:
# Settings.

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "benchmark" else Path.cwd()
EMBEDDING_H5 = PROJECT_ROOT / "artifacts/embeddings/experiment_a_image_only_embeddings.h5"

TARGET_LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Pleural Effusion",
]

SOLVER = "lbfgs"
MAX_ITER = 1000
RANDOM_STATE = 42
BENCHMARK_LAYER = 20
PARALLEL_CONFIGS = [
    {"n_jobs": 1, "inner_threads": None},
    {"n_jobs": 4, "inner_threads": 32},
    {"n_jobs": 8, "inner_threads": 16},
    {"n_jobs": 16, "inner_threads": 8},
]

print(EMBEDDING_H5, EMBEDDING_H5.exists())
print("Primary metric: AUROC. AUPRC is secondary and should be interpreted against positive prevalence.")


In [ ]:
# Load metadata and labels.

with h5py.File(EMBEDDING_H5, "r") as h5:
    labels = h5["labels"][:]
    probe_split = h5["probe_split"][:].astype(str)
    print("labels", labels.shape)
    print("probe_split", pd.Series(probe_split).value_counts().to_dict())
    print("medsiglip/global", h5["medsiglip/global"].shape, h5["medsiglip/global"].dtype)
    print("medgemma/projected_image_mean", h5["medgemma/projected_image_mean"].shape, h5["medgemma/projected_image_mean"].dtype)
    print("medgemma/layer_image_mean", h5["medgemma/layer_image_mean"].shape, h5["medgemma/layer_image_mean"].dtype)
    print("medgemma/layer_last_image", h5["medgemma/layer_last_image"].shape, h5["medgemma/layer_last_image"].dtype)

train_idx = np.where(probe_split == "train")[0]
test_idx = np.where(probe_split == "test")[0]

print("train", len(train_idx))
print("test", len(test_idx))


In [ ]:
# Helper functions.

def current_rss_gb():
    status_path = Path("/proc/self/status")
    if not status_path.exists():
        return np.nan
    for line in status_path.read_text().splitlines():
        if line.startswith("VmRSS:"):
            return float(line.split()[1]) / 1024**2
    return np.nan


def load_feature_matrix(dataset_name, layer=None):
    rss_before = current_rss_gb()
    t0 = perf_counter()
    c0 = process_time()
    with h5py.File(EMBEDDING_H5, "r") as h5:
        if layer is None:
            x_train = h5[dataset_name][train_idx].astype(np.float32)
            x_test = h5[dataset_name][test_idx].astype(np.float32)
        else:
            x_train = h5[dataset_name][train_idx, layer, :].astype(np.float32)
            x_test = h5[dataset_name][test_idx, layer, :].astype(np.float32)

    train_finite = np.isfinite(x_train)
    test_finite = np.isfinite(x_test)
    if not train_finite.all() or not test_finite.all():
        layer_text = "" if layer is None else f" layer={layer}"
        raise ValueError(
            f"Non-finite values in {dataset_name}{layer_text}: "
            f"train_bad={int(x_train.size - train_finite.sum())}, "
            f"test_bad={int(x_test.size - test_finite.sum())}. "
            "Rerun benchmark_embeddings.ipynb or regenerate the benchmark H5 with MedGemma features saved as float32."
        )

    return x_train, x_test, {
        "feature_load_wall_sec": perf_counter() - t0,
        "feature_load_cpu_sec": process_time() - c0,
        "feature_load_rss_before_gb": rss_before,
        "feature_load_rss_after_gb": current_rss_gb(),
    }


def make_probe():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            solver=SOLVER,
            max_iter=MAX_ITER,
            random_state=RANDOM_STATE,
        ),
    )


def fit_one_label(x_train, x_test, label_i, inner_threads=None):
    label_name = TARGET_LABELS[label_i]
    y_train = labels[train_idx, label_i]
    y_test = labels[test_idx, label_i]
    model = make_probe()

    rss_before = current_rss_gb()
    t0 = perf_counter()
    c0 = process_time()
    if inner_threads is None:
        model.fit(x_train, y_train)
    else:
        with threadpool_limits(limits=inner_threads):
            model.fit(x_train, y_train)
    train_time = perf_counter() - t0
    cpu_time = process_time() - c0
    rss_after = current_rss_gb()

    scores = model.predict_proba(x_test)[:, 1]
    positive_prevalence = float(y_test.mean())
    auroc = roc_auc_score(y_test, scores) if len(np.unique(y_test)) == 2 else np.nan
    auprc = average_precision_score(y_test, scores)

    return {
        "label": label_name,
        "solver": SOLVER,
        "positive_prevalence": positive_prevalence,
        "train_time_sec": train_time,
        "train_cpu_sec": cpu_time,
        "fit_rss_before_gb": rss_before,
        "fit_rss_after_gb": rss_after,
        "auroc": auroc,
        "auprc": auprc,
        "n_iter": int(model.named_steps["logisticregression"].n_iter_[0]),
    }


def train_five_label_probes(x_train, x_test):
    rows = [fit_one_label(x_train, x_test, label_i) for label_i in range(len(TARGET_LABELS))]
    total_train_time = sum(row["train_time_sec"] for row in rows)
    total_cpu_time = sum(row["train_cpu_sec"] for row in rows)
    return rows, total_train_time, total_cpu_time


def benchmark_feature(feature_name, dataset_name, layer=None):
    x_train, x_test, load_stats = load_feature_matrix(dataset_name, layer=layer)
    print(feature_name, "x_train", x_train.shape, "x_test", x_test.shape, "load_sec", round(load_stats["feature_load_wall_sec"], 2))

    rows, train_time, cpu_time = train_five_label_probes(x_train, x_test)
    for row in rows:
        row["feature"] = feature_name
        row["feature_dim"] = x_train.shape[1]
        row["five_label_train_time_sec"] = train_time
        row["five_label_train_cpu_sec"] = cpu_time
        row.update(load_stats)

    del x_train, x_test
    return pd.DataFrame(rows)



def train_feature_job(job, inner_threads=None):
    x_train, x_test, load_stats = load_feature_matrix(job["dataset_name"], layer=job.get("layer"))
    t0 = perf_counter()
    rows = [fit_one_label(x_train, x_test, label_i, inner_threads=inner_threads) for label_i in range(len(TARGET_LABELS))]
    wall_time = perf_counter() - t0

    result = {
        "feature": job["feature"],
        "feature_dim": x_train.shape[1],
        "five_label_wall_sec": wall_time,
        "mean_positive_prevalence": float(np.mean([row["positive_prevalence"] for row in rows])),
        "mean_auroc": float(np.mean([row["auroc"] for row in rows])),
        "mean_auprc": float(np.mean([row["auprc"] for row in rows])),
        "max_n_iter": int(np.max([row["n_iter"] for row in rows])),
        **load_stats,
    }

    del x_train, x_test
    return result


def benchmark_parallel_jobs(jobs):
    rows = []
    for config in PARALLEL_CONFIGS:
        n_jobs = config["n_jobs"]
        inner_threads = config["inner_threads"]

        t0 = perf_counter()
        if n_jobs == 1:
            job_results = [train_feature_job(job, inner_threads=inner_threads) for job in jobs]
        else:
            job_results = Parallel(n_jobs=n_jobs)(
                delayed(train_feature_job)(job, inner_threads=inner_threads)
                for job in jobs
            )
        wall_time = perf_counter() - t0

        rows.append({
            "n_jobs": n_jobs,
            "inner_threads": inner_threads if inner_threads is not None else "default",
            "num_feature_jobs": len(jobs),
            "wall_time_sec": wall_time,
            "estimated_experiment_a_wall_min": wall_time * (70 / len(jobs)) / 60,
            "mean_auroc": float(np.mean([row["mean_auroc"] for row in job_results])),
            "mean_auprc": float(np.mean([row["mean_auprc"] for row in job_results])),
            "rss_after_gb": current_rss_gb(),
        })

    return pd.DataFrame(rows)


In [ ]:
# Benchmark representative feature matrices.

results = []

results.append(benchmark_feature(
    feature_name="medsiglip_global",
    dataset_name="medsiglip/global",
))

results.append(benchmark_feature(
    feature_name="medgemma_projected_image_mean",
    dataset_name="medgemma/projected_image_mean",
))

results.append(benchmark_feature(
    feature_name="medgemma_layer_image_mean_layer20",
    dataset_name="medgemma/layer_image_mean",
    layer=BENCHMARK_LAYER,
))

results.append(benchmark_feature(
    feature_name="medgemma_layer_last_image_layer20",
    dataset_name="medgemma/layer_last_image",
    layer=BENCHMARK_LAYER,
))

benchmark_df = pd.concat(results, ignore_index=True)
display(benchmark_df)


In [ ]:
# Summarize probe speed and estimate full Experiment A runtime.

summary = (
    benchmark_df
    .groupby(["feature", "feature_dim"], as_index=False)
    .agg(
        feature_load_wall_sec=("feature_load_wall_sec", "first"),
        feature_load_rss_after_gb=("feature_load_rss_after_gb", "first"),
        five_label_train_time_sec=("five_label_train_time_sec", "first"),
        mean_label_train_time_sec=("train_time_sec", "mean"),
        max_fit_rss_after_gb=("fit_rss_after_gb", "max"),
        mean_positive_prevalence=("positive_prevalence", "mean"),
        mean_auroc=("auroc", "mean"),
        mean_auprc=("auprc", "mean"),
        max_n_iter=("n_iter", "max"),
    )
)

display(summary)

medsiglip_time = summary.query("feature == 'medsiglip_global'")["five_label_train_time_sec"].iloc[0]
projected_time = summary.query("feature == 'medgemma_projected_image_mean'")["five_label_train_time_sec"].iloc[0]
layer_mean_time = summary.query("feature == 'medgemma_layer_image_mean_layer20'")["five_label_train_time_sec"].iloc[0]
layer_last_time = summary.query("feature == 'medgemma_layer_last_image_layer20'")["five_label_train_time_sec"].iloc[0]

estimated_train_time = medsiglip_time + projected_time + 34 * layer_mean_time + 34 * layer_last_time
estimate_df = pd.DataFrame([{
    "solver": SOLVER,
    "estimated_experiment_a_train_time_sec": estimated_train_time,
    "estimated_experiment_a_train_time_min": estimated_train_time / 60,
}])
display(estimate_df)


In [ ]:
# Benchmark process parallelism across feature/layer jobs.

parallel_jobs = [
    {"feature": "medsiglip_global", "dataset_name": "medsiglip/global"},
    {"feature": "medgemma_projected_image_mean", "dataset_name": "medgemma/projected_image_mean"},
    {"feature": "medgemma_layer_image_mean_layer0", "dataset_name": "medgemma/layer_image_mean", "layer": 0},
    {"feature": "medgemma_layer_image_mean_layer10", "dataset_name": "medgemma/layer_image_mean", "layer": 10},
    {"feature": "medgemma_layer_image_mean_layer20", "dataset_name": "medgemma/layer_image_mean", "layer": 20},
    {"feature": "medgemma_layer_image_mean_layer33", "dataset_name": "medgemma/layer_image_mean", "layer": 33},
    {"feature": "medgemma_layer_last_image_layer0", "dataset_name": "medgemma/layer_last_image", "layer": 0},
    {"feature": "medgemma_layer_last_image_layer10", "dataset_name": "medgemma/layer_last_image", "layer": 10},
    {"feature": "medgemma_layer_last_image_layer20", "dataset_name": "medgemma/layer_last_image", "layer": 20},
    {"feature": "medgemma_layer_last_image_layer33", "dataset_name": "medgemma/layer_last_image", "layer": 33},
]

parallel_df = benchmark_parallel_jobs(parallel_jobs)
display(parallel_df)
